In [1]:
#from google.colab import drive
#drive.mount('/content/drive')
#print("hi")

Mounted at /content/drive


This initial cell sets up the environment and defines the core configurations for your model training.

**1. Dependency Verification and Installation:**
   - It checks for the presence of essential Python libraries (like `numpy`, `torch`, `musdb`, `nbformat`, etc.).
   - If any required library is not found, it attempts to install it automatically using `pip`.
   - It also verifies the availability of PyTorch with CUDA, which is crucial for GPU-accelerated training.

**2. Experiment Configurations:**
   - `MODEL_CONFIG`: Defines the neural network architecture parameters (e.g., number of input/output channels, base filters, number of layers, batch normalization, dropout).
   - `TRAIN_CONFIG`: Specifies the hyperparameters for the main training loop (e.g., number of epochs, learning rate, patience for early stopping, batch size).
   - `OVERFIT_CONFIG`: Provides a separate set of hyperparameters specifically designed for an overfit test, often used to ensure the model can learn at all on a small subset of data.

In [2]:
# --- 1. Verify and install dependencies ---
print("Verifying and installing missing packages if necessary...")

packages_to_check = [
    'numpy', 'matplotlib', 'librosa', 'tqdm', 'sklearn', 'stempeg', 'torch', 'torchvision', 'torchaudio', 'musdb', 'nbformat', 'nbconvert'
]

for package in packages_to_check:
    try:
        __import__(package)
        print(f"  ✅ {package} is installed.")
    except ImportError:
        print(f"  ❌ {package} is NOT installed. Attempting to install...")
        try:
            import sys
            import subprocess
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])
            __import__(package)
            print(f"  ✅ {package} is now installed.")
        except Exception as e:
            print(f"  ❌ Failed to install {package}: {e}")

# Special check for PyTorch CUDA
print("\n--- PyTorch CUDA status ---")
try:
    import torch
    if torch.cuda.is_available():
        print(f"  ✅ PyTorch with CUDA (version {torch.version.cuda}) is available.")
        print(f"     CUDA Device Name: {torch.cuda.get_device_name(0)}")
    else:
        print("  ⚠️ PyTorch is installed, but CUDA is NOT available.")
except ImportError:
    print("  ❌ PyTorch is not installed.")

print("Verification complete.")

# --- 2. Set experiment configs for model_A.ipynb ---
MODEL_CONFIG = {
    'in_channels': 1,
    'out_channels': 1,
    'base_filters': 64,
    'num_layers': 4,
    'batchnorm': True,
    'dropout': 0.1,
}
TRAIN_CONFIG = {
    'num_epochs': 50,
    'learning_rate': 1e-4,
    'patience': 6, # how many epochs with no major loss change
    'batch_size': 2,
}
OVERFIT_CONFIG = {
    'base_filters': 128,
    'num_layers': 4,
    'batchnorm': True,
    'dropout': 0.0,
    'learning_rate': 3e-4,
    'num_epochs': 100,
    'batch_size': 2,
    'patience': 6, # how many epochs with no major loss change
}


Verifying and installing missing packages if necessary...
  ✅ numpy is installed.
  ✅ matplotlib is installed.
  ✅ librosa is installed.
  ✅ tqdm is installed.
  ✅ sklearn is installed.
  ❌ stempeg is NOT installed. Attempting to install...
  ✅ stempeg is now installed.
  ✅ torch is installed.
  ✅ torchvision is installed.
  ✅ torchaudio is installed.
  ❌ musdb is NOT installed. Attempting to install...
  ✅ musdb is now installed.
  ✅ nbformat is installed.
  ✅ nbconvert is installed.

--- PyTorch CUDA status ---
  ✅ PyTorch with CUDA (version 12.6) is available.
     CUDA Device Name: Tesla T4
Verification complete.


# Main Experiment Controller

This notebook lets you control model architecture and training hyperparameters for model_A.ipynb from a single place.

**Workflow:**
1. Set your experiment configs below (MODEL_CONFIG, TRAIN_CONFIG, OVERFIT_CONFIG).
2. Run all cells to verify dependencies and execute model_A.ipynb with your chosen parameters.
3. To try different experiments, just change the config values and re-run.

**Example:**
- Change `base_filters` or `batch_size` in the config dicts to test different model sizes or memory usage.
- All results, checkpoints, and logs will be produced as usual by model_A.ipynb.


In [3]:
import os
import nbformat
from nbconvert.preprocessors import ExecutePreprocessor

# --- 3. Find and run model_A.ipynb with injected configs ---

def find_notebook(filename, search_path="."):
    for root, dirs, files in os.walk(search_path):
        if filename in files:
            return os.path.join(root, filename)
    return None

notebook_path = find_notebook("model_A.ipynb", ".")
if notebook_path is None:
    raise FileNotFoundError("model_A.ipynb not found anywhere in the workspace.")

with open(notebook_path) as f:
    nb = nbformat.read(f, as_version=4)

# Remove autoreload magic commands from the notebook before execution
for cell in nb.cells:
    if cell.cell_type == 'code':
        lines = cell.source.splitlines()
        new_lines = [line for line in lines if not line.strip().startswith(('%load_ext autoreload', '%autoreload'))]
        cell.source = '\n'.join(new_lines)

# Do NOT inject config variables; configs are set in main.ipynb and available in the execution environment

# Execute the notebook
ep = ExecutePreprocessor(timeout=1200, kernel_name='python3')
try:
    ep.preprocess(nb, {'metadata': {'path': os.path.dirname(notebook_path)}})
    print(f'model_A.ipynb executed successfully from: {notebook_path}')
except Exception as e:
    print(f'Error executing model_A.ipynb: {e}')

# to monitor the loss progress in terminal run:
# watch -n 10 python show_latest_epoch.py


KeyboardInterrupt: 